# TurkishBERTweet - Siber Zorbalık Sınıflandırması
Bu notebook, **TurkishBERTweet** modeli ile siber zorbalık tespiti yapmayı hedefler.

Adil bir kıyaslama için DOCX makalesindeki kriterlere **birebir uygun** olarak kodlanmıştır:
- Epoch: 4
- Learning Rate: 2e-5
- Batch Size: 16
- Max Sequence Length: 256
- Eğitim / Test Ayrımı: %80 / %20

In [12]:
import pandas as pd
import numpy as np
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'*** Kullanılan Cihaz: {device} ***')

*** Kullanılan Cihaz: cuda ***


In [13]:
# -----------------------------------------------------
# 1. MODEL VE HİPERPARAMETRELER (Orijinal Makaleye Göre)
# -----------------------------------------------------
EPOCHS = 4
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_LEN = 256
MODEL_NAME = 'VRLLab/TurkishBERTweet'
RANDOM_STATE = 42

In [14]:
# -----------------------------------------------------
# 2. VERİ SETİ (DATASET) SINIFI
# -----------------------------------------------------
class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

In [15]:
# -----------------------------------------------------
# 3. EĞİTİM VE TEST FONKSİYONLARI
# -----------------------------------------------------
import copy
from transformers import get_linear_schedule_with_warmup

def train_epoch(model, data_loader, optimizer, device, n_examples, scheduler):
    model = model.train()
    losses = []
    correct_predictions = 0
    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['targets'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
        loss = outputs.loss
        logits = outputs.logits
        _, preds = torch.max(logits, dim=1)
        
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    real_targets, predictions = [], []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['targets'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss
            logits = outputs.logits
            _, preds = torch.max(logits, dim=1)
            
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            real_targets.extend(targets.cpu().numpy())
            predictions.extend(preds.cpu().numpy())
            
    accuracy = accuracy_score(real_targets, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(real_targets, predictions, average='macro', zero_division=0)
    return accuracy, precision, recall, f1, np.mean(losses)

def run_experiment(file_path):
    print(f'\n========== [ MESAİ BAŞLADI: {file_path} ] ==========')
    df = pd.read_excel(file_path)
    
    text_col = df.columns[0]
    label_col = 'total' if 'total' in df.columns else df.columns[1]
    df = df.dropna(subset=[text_col, label_col])
    
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[label_col])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_data_loader = DataLoader(CyberbullyingDataset(df_train[text_col].to_numpy(), df_train[label_col].to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    test_data_loader = DataLoader(CyberbullyingDataset(df_test[text_col].to_numpy(), df_test[label_col].to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    
    # 🌟 YÖNTEM 2: Weight Decay Optimizasyonu
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)
    
    # 🌟 YÖNTEM 3: Linear Scheduler with Warmup
    total_steps = len(train_data_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
    
    best_f1 = 0
    best_model_weights = None
    
    for epoch in range(EPOCHS):
        print(f'Epoch {epoch + 1}/{EPOCHS} çalışıyor...')
        train_acc, train_loss = train_epoch(model, train_data_loader, optimizer, device, len(df_train), scheduler)
        
        # 🌟 YÖNTEM 1: Best Checkpoint (Validation her epoch sonunda yapılıyor)
        val_acc, val_precision, val_recall, val_f1, val_loss = eval_model(model, test_data_loader, device, len(df_test))
        print(f'  ✓ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test F1: {val_f1:.4f}')
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_weights = copy.deepcopy(model.state_dict())
            
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print("💡 En iyi Epoch ağırlıkları geri yüklendi!")
        
    accuracy, precision, recall, f1, test_loss = eval_model(model, test_data_loader, device, len(df_test))
    
    print(f'\n🔥 FİNAL GÜNCELLENMİŞ SONUÇ METRİKLERİ ({file_path}):')
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("="*60)
    
    del model, optimizer, scheduler, train_data_loader, test_data_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return accuracy, precision, recall, f1


In [16]:
# --- ÇALIŞTIRMA BÖLÜMÜ ---
# Kendi yerel projenizdeki dosyaları bulup sırasıyla eğitir. 
pre_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesi.xlsx')
post_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidsonrasi.xlsx')
combined_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesivesonrasi.xlsx')


========== [ MESAİ BAŞLADI: /content/drive/MyDrive/data/covidoncesi.xlsx ] ==========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 çalışıyor...
  ✓ Train Loss: 0.5746 | Train Acc: 0.7219 | Test F1: 0.7302
Epoch 2/4 çalışıyor...
  ✓ Train Loss: 0.4173 | Train Acc: 0.8302 | Test F1: 0.7291
Epoch 3/4 çalışıyor...
  ✓ Train Loss: 0.2053 | Train Acc: 0.9254 | Test F1: 0.7258
Epoch 4/4 çalışıyor...
  ✓ Train Loss: 0.0889 | Train Acc: 0.9700 | Test F1: 0.6948
💡 En iyi Epoch ağırlıkları geri yüklendi!

🔥 FİNAL GÜNCELLENMİŞ SONUÇ METRİKLERİ (/content/drive/MyDrive/data/covidoncesi.xlsx):
Accuracy:  0.7911
Precision: 0.7840
Recall:    0.7124
F1-Score:  0.7302

========== [ MESAİ BAŞLADI: /content/drive/MyDrive/data/covidsonrasi.xlsx ] ==========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 çalışıyor...
  ✓ Train Loss: 0.6425 | Train Acc: 0.6397 | Test F1: 0.5965
Epoch 2/4 çalışıyor...
  ✓ Train Loss: 0.5377 | Train Acc: 0.7398 | Test F1: 0.5952
Epoch 3/4 çalışıyor...
  ✓ Train Loss: 0.2736 | Train Acc: 0.8900 | Test F1: 0.5918
Epoch 4/4 çalışıyor...
  ✓ Train Loss: 0.1139 | Train Acc: 0.9580 | Test F1: 0.5986
💡 En iyi Epoch ağırlıkları geri yüklendi!

🔥 FİNAL GÜNCELLENMİŞ SONUÇ METRİKLERİ (/content/drive/MyDrive/data/covidsonrasi.xlsx):
Accuracy:  0.6424
Precision: 0.6014
Recall:    0.5971
F1-Score:  0.5986

========== [ MESAİ BAŞLADI: /content/drive/MyDrive/data/covidoncesivesonrasi.xlsx ] ==========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 çalışıyor...
  ✓ Train Loss: 0.6050 | Train Acc: 0.6933 | Test F1: 0.6322
Epoch 2/4 çalışıyor...
  ✓ Train Loss: 0.4712 | Train Acc: 0.7879 | Test F1: 0.6548
Epoch 3/4 çalışıyor...
  ✓ Train Loss: 0.2111 | Train Acc: 0.9157 | Test F1: 0.6152
Epoch 4/4 çalışıyor...
  ✓ Train Loss: 0.0766 | Train Acc: 0.9734 | Test F1: 0.6206
💡 En iyi Epoch ağırlıkları geri yüklendi!

🔥 FİNAL GÜNCELLENMİŞ SONUÇ METRİKLERİ (/content/drive/MyDrive/data/covidoncesivesonrasi.xlsx):
Accuracy:  0.7040
Precision: 0.6654
Recall:    0.6498
F1-Score:  0.6548
